# RAG System with LLM Judge for Benchmarking
This notebook implements a RAG system using MiniLM and FAISS, with Mistral-7B as both the response generator and judge for benchmarking purposes.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")

torch.cuda.set_device(0)

# Essential imports
import json
from datetime import datetime
from tqdm import tqdm
import subprocess

# LangChain and ML imports
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

3


DeferredCudaCallError: CUDA call failed lazily at initialization with error: device >= 0 && device < num_gpus INTERNAL ASSERT FAILED at "../aten/src/ATen/cuda/CUDAContext.cpp":50, please report a bug to PyTorch. 

CUDA call was originally invoked at:

['  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/runpy.py", line 197, in _run_module_as_main\n    return _run_code(code, main_globals, None,\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/runpy.py", line 87, in _run_code\n    exec(code, run_globals)\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>\n    app.launch_new_instance()\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/traitlets/config/application.py", line 1077, in launch_instance\n    app.start()\n', '  File "/home/synthesisproject/anaconda3/envs/soroush1/lib/python3.9/site-packages/ipykernel/kernelapp.py", lin

## Load and Process JSON Records

In [2]:
# Load JSON records from file
with open('filtered_records_on_zeolite.json', 'r') as file:
    filtered_records = json.load(file)

# Process records into documents
documents = []
metadata = []

for record in filtered_records:
    doi = record.get("doi", "Unknown DOI")
    
    # Add abstract as a separate document
    abstract_text = record.get("abstract", "").strip()
    if abstract_text:
        combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
        documents.append(combined_text)
        metadata.append({"doi": doi, "source": "abstract"})
    
    # Add each paragraph as a separate document
    for para in record.get("paragraphs", []):
        paragraph_text = para.get("text", "").strip()
        if paragraph_text:
            combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
            documents.append(combined_text)
            metadata.append({"doi": doi, "source": "paragraph"})

print(f"Total documents processed: {len(documents)}")

Total documents processed: 1474439


## Create Vector Database

In [ ]:
# Initialize embeddings model
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# TOGGLE: Set to True for testing (2 batches), False for full processing
TESTING_MODE = True

# Create FAISS vector store with batch processing
batch_size = 50
vector_db = None
max_batches = 2 if TESTING_MODE else None  # 2 batches for testing, None for all

for i, batch_start in enumerate(range(0, len(documents), batch_size)):
    if max_batches and i >= max_batches:
        break
        
    docs_batch = documents[batch_start:batch_start + batch_size]
    meta_batch = metadata[batch_start:batch_start + batch_size]
    
    print(f"Processing batch {i + 1}, documents {batch_start} to {batch_start + len(docs_batch)}")
    
    # Show sample content in testing mode
    if TESTING_MODE and i < 2:
        print(f"Sample from batch {i + 1}:")
        sample_doc = docs_batch[0]
        print(f"  Source: {meta_batch[0]['source']}")
        print(f"  Preview: {sample_doc[:150]}...")
        print()
    
    if vector_db is None:
        vector_db = FAISS.from_texts(docs_batch, embedding=embeddings, metadatas=meta_batch)
    else:
        vector_db.add_texts(docs_batch, metadatas=meta_batch)

print(f"Vector database created with {vector_db.index.ntotal} vectors")

Processing batch 1, documents 0 to 50
Sample from batch 1:
  Source: abstract
  Preview: Synthesis of ZSM-5 from template-free batches which preceded the preparation of template-free ZSM-5 layers on porous supports was studied to ascertain...

Processing batch 2, documents 50 to 100
Sample from batch 2:
  Source: paragraph
  Preview: The alkali treatment causes a significant change on the surface of the MFI zeolite. As shown in Fig. 2, many small holes with diameters of about 10nm ...

Vector database created with 100 vectors
Processing batch 2, documents 50 to 100
Sample from batch 2:
  Source: paragraph
  Preview: The alkali treatment causes a significant change on the surface of the MFI zeolite. As shown in Fig. 2, many small holes with diameters of about 10nm ...

Vector database created with 100 vectors


## Setup RAG and Judge Models

# Model-specific imports and GPU management

In [1]:
# Model-specific imports
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def kill_soroush1_processes():
    """Kill any existing soroush1 processes to free GPU memory"""
    try:
        ps_output = subprocess.check_output("ps aux | grep soroush1/bin/python", shell=True).decode()
        for line in ps_output.split('\n'):
            if '/envs/soroush1/bin/python' in line and 'grep' not in line:
                pid = line.split()[1]
                print(f"Killing soroush1 process with PID: {pid}")
                subprocess.run(f"kill -9 {pid}", shell=True)
    except subprocess.CalledProcessError:
        print("No soroush1 processes found")

def cleanup_gpu():
    """Cleanup GPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("GPU memory cleared")

# Show current GPU status
!nvidia-smi


Mon Sep 15 22:37:44 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   31C    P8    21W / 230W |  18889MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

In [ ]:
# Model configuration
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Initialize tokenizer and model for RAG responses
rag_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, 
                                             use_auth_token=hf_token)
rag_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, 
                                                use_auth_token=hf_token, 
                                                torch_dtype=torch.float16)

# Create pipeline for RAG
rag_pipeline = pipeline("text-generation", model=rag_model, tokenizer=rag_tokenizer, max_length=10000)
rag_llm = HuggingFacePipeline(pipeline=rag_pipeline)

# Initialize separate model instance for judging (using same model)
judge_pipeline = pipeline("text-generation", model=rag_model, tokenizer=rag_tokenizer, max_length=2000)
judge_llm = HuggingFacePipeline(pipeline=judge_pipeline)

## Define Prompts

In [ ]:
# RAG prompt template
RAG_PROMPT = """
Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""

# Judge prompt template
JUDGE_PROMPT = """
You are an expert judge evaluating the quality of an answer generated by an AI system.
You will be provided with:
1. The original question
2. The context provided to the AI
3. The AI's answer

Please evaluate the answer on the following criteria:
1. Relevance (0-10): How well does the answer address the question?
2. Accuracy (0-10): How accurate is the answer based on the provided context?
3. Completeness (0-10): How complete is the answer?
4. Citation (0-10): Does it properly cite the DOI?

Question: {question}
Context: {context}
Answer: {answer}

Provide your evaluation in JSON format with scores and brief explanations:
{
    "relevance": {"score": X, "explanation": "..."}, 
    "accuracy": {"score": X, "explanation": "..."},
    "completeness": {"score": X, "explanation": "..."},
    "citation": {"score": X, "explanation": "..."},
    "total_score": X,
    "overall_feedback": "..."
}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
judge_prompt = ChatPromptTemplate.from_template(JUDGE_PROMPT)

## Evaluation Function

In [ ]:
def evaluate_rag_response(query, k=1):
    # Get relevant documents
    retrieved_docs = vector_db.similarity_search(query, k=k)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Generate RAG response
    rag_chain = LLMChain(llm=rag_llm, prompt=rag_prompt)
    response = rag_chain.run(context=context_text, question=query)
    
    # Generate evaluation
    judge_chain = LLMChain(llm=judge_llm, prompt=judge_prompt)
    evaluation = judge_chain.run(
        question=query,
        context=context_text,
        answer=response
    )
    
    # Parse the evaluation (assuming it's valid JSON)
    try:
        evaluation_dict = json.loads(evaluation)
    except json.JSONDecodeError:
        evaluation_dict = {
            "error": "Failed to parse evaluation",
            "raw_evaluation": evaluation
        }
    
    return {
        "query": query,
        "context": context_text,
        "response": response,
        "evaluation": evaluation_dict,
        "timestamp": datetime.now().isoformat()
    }

## Run Evaluation

In [ ]:
# List of test queries
test_queries = [
    "How is ZSM-5 synthesized?",
    "What is the best way to synthesize hierarchical ZSM-5?",
    "How is Silicalite-1 synthesized?",
    "What are the typical synthesis conditions for ZSM-5?"
]

# Run evaluation for each query
results = []
for query in tqdm(test_queries, desc="Evaluating queries"):
    result = evaluate_rag_response(query)
    results.append(result)

# Save results to JSON file
output_filename = f"rag_evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_filename}")

## Analysis of Results

In [ ]:
# Calculate average scores
def analyze_results(results):
    total_scores = {
        'relevance': 0,
        'accuracy': 0,
        'completeness': 0,
        'citation': 0,
        'total_score': 0
    }
    valid_results = 0
    
    for result in results:
        eval_dict = result['evaluation']
        if 'error' not in eval_dict:
            valid_results += 1
            total_scores['relevance'] += eval_dict['relevance']['score']
            total_scores['accuracy'] += eval_dict['accuracy']['score']
            total_scores['completeness'] += eval_dict['completeness']['score']
            total_scores['citation'] += eval_dict['citation']['score']
            total_scores['total_score'] += eval_dict['total_score']
    
    if valid_results > 0:
        avg_scores = {k: v/valid_results for k, v in total_scores.items()}
        print("\nAverage Scores:")
        for metric, score in avg_scores.items():
            print(f"{metric}: {score:.2f}")
    else:
        print("No valid evaluations found")

analyze_results(results)